#### Attention — companion notebook

Companion for [`attention.md`](./attention.md). We build $Q, K, V$ from a tiny toy sequence, compute scaled-dot-product attention, and visualize the attention matrix with and without causal masking.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

In [ ]:
n, d, dk = 6, 8, 4  # sequence length, embedding dim, projected key dim
X = rng.standard_normal((n, d))
Wq, Wk, Wv = rng.standard_normal((d, dk))*0.3, rng.standard_normal((d, dk))*0.3, rng.standard_normal((d, dk))*0.3
Q, K, V = X @ Wq, X @ Wk, X @ Wv

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    logits = Q @ K.T / np.sqrt(K.shape[-1])
    if mask is not None:
        logits = logits + mask
    A = softmax(logits, axis=-1)
    return A @ V, A

# Causal mask: position i can't attend to j > i
mask = np.zeros((n, n))
mask[np.triu_indices(n, k=1)] = -1e9

_, A_full = attention(Q, K, V)
_, A_causal = attention(Q, K, V, mask=mask)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, A, title in zip(axes, [A_full, A_causal], ['Full self-attention', 'Causal-masked self-attention']):
    im = ax.imshow(A, vmin=0, vmax=1, cmap='Blues')
    ax.set_xlabel('key position'); ax.set_ylabel('query position'); ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.show()
print('Row sums (should be 1):', A_full.sum(axis=-1).round(4))
print('Causal row sums       :', A_causal.sum(axis=-1).round(4))